# 第 25 章：信用衍生品与 CDS 市场

本章的全部数字：从基本原理定价，到带违约的树上的可转债与混合型票据。

| 节 | 复现什么 | 例 |
|---|---|---|
| 1 | 由报价给 CDS 定价，并把平价利差反推回来 | 25.1 |
| 2 | 从利差到期初支付，以及年金为什么变短 | 25.2 |
| 3 | 给头寸估值：年金也在变 | 25.3 |
| 4 | 指数按年金加权平均而不是算术平均交易 | 25.4 |
| 5 | 同一个资产池，五个分层，一个相关性 | 25.5 |
| 6 | 同样五个主体，五种篮子工具 | 25.6 |
| 7 | 重组条款值多少钱 | 25.7 |
| 8 | 分解一只可转债 | 25.8 |
| 9 | 带违约的树上的可转债：转股、发行人赎回与 Merton 的跳跃至破产 | 25.9 |
| 10 | 混合型结构需要、而市场没有报价的相关性 | 25.10 |

全章都是确定性的：没有任何模拟，也没有随机种子。第 5、6 节对单因子高斯
copula 关于共同因子的积分，用的是细网格数值积分而不是抽样，因此每次运行给出完全相同的位数。

表头与传给 `show()` 的标签保留英文：中文字符在等宽终端里是双宽的，而 Python 的
`%-50s` 按字符数计，翻译一行就会把下面每一行都错开。


In [1]:
import numpy as np
from math import exp, log, sqrt, tanh
from scipy.stats import norm, binom

N = norm.cdf
R, RATE, FREQ, TENOR = 0.40, 0.03, 4, 5.0   # 回收率、贴现率、每年付息次数、期限
NOTIONAL = 10_000_000.0


def show(label, got, printed, tol=6e-4):
    """把算出来的值印在本章所印的值旁边。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-46s %13.4f   book %11.4f   %s" % (label, got, printed, flag))


def cds_legs(lam, T=TENOR, R=R, r=RATE, freq=FREQ, accrual=True, pay_at_default=False):
    """平坦违约强度下的风险年金与保护端。

    返回 (年金, 保护端, 排定部分, 应计部分)。本章的惯例把违约支付贴现到违约
    所在季度的期末；pay_at_default 把它改到季度中点，而后者才是合约真正的
    约定，两者相差百分之几。
    """
    n, dt = int(round(T * freq)), 1.0 / freq
    sched = accr = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        Q, Q_prev = exp(-lam * t), exp(-lam * t_prev)
        dQ = Q_prev - Q
        df = exp(-r * t)
        sched += dt * df * Q
        if accrual:
            accr += 0.5 * dt * df * dQ
        prot += (exp(-r * (t - dt / 2)) if pay_at_default else df) * dQ
    return sched + accr, (1 - R) * prot, sched, accr


## 1. 从头给一笔 CDS 定价：例 25.1

对一个报价 250 个基点的主体买入 1,000 万美元、五年期的保护，按标准的 100 个基点
票息成交，回收率 40%，利率 3%。

信用三角把报价变成违约强度，两端随之而出，平价利差又回到原处。
**它能几乎精确地回来，是惯例的性质，而不是一条定理**：起作用的是那项应计，
而它在 4.1865 里只值 0.0218。


In [2]:
s_par, coupon = 0.0250, 0.0100
lam = s_par / (1 - R)
print("hazard rate  lambda = s/(1-R) = %.3f%%" % (100 * lam))
print()
print("%6s %10s %22s" % ("year", "Q(t)", "cumulative default"))
for t in (1, 2, 3, 5):
    Q = exp(-lam * t)
    print("%6d %10.4f %21.2f%%" % (t, Q, 100 * (1 - Q)))
print()

A, prot, sched, accr = cds_legs(lam)
show("scheduled premium leg", sched, 4.1647)
show("accrual on default", accr, 0.0218)
show("risky annuity A", A, 4.1865)
show("protection leg, % of notional", 100 * prot, 10.4662)
show("par spread recovered, bp", 1e4 * prot / A, 250.00)
print()

dt = 1.0 / FREQ
show("closed form 2(1-R)/dt tanh(lam dt/2), bp",
     1e4 * 2 * (1 - R) / dt * tanh(lam * dt / 2), 249.998)
_, prot_d, _, _ = cds_legs(lam, pay_at_default=True)
show("protection paid at default, % of notional", 100 * prot_d, 10.5056)
print()

upfront = 100 * prot - 100 * coupon * A
show("upfront, legs differenced, %", upfront, 6.2797)
show("upfront, short form (s-c)A, %", 100 * (s_par - coupon) * A, 6.2798)
print("按 1,000 万美元名义本金计为 $%s" % format(NOTIONAL * upfront / 100, ",.0f"))


hazard rate  lambda = s/(1-R) = 4.167%

  year       Q(t)     cumulative default
     1     0.9592                  4.08%
     2     0.9200                  8.00%
     3     0.8825                 11.75%
     5     0.8119                 18.81%

scheduled premium leg                                 4.1647   book      4.1647   ok
accrual on default                                    0.0218   book      0.0218   ok
risky annuity A                                       4.1865   book      4.1865   ok
protection leg, % of notional                        10.4662   book     10.4662   ok
par spread recovered, bp                            249.9977   book    250.0000   ok

closed form 2(1-R)/dt tanh(lam dt/2), bp            249.9977   book    249.9980   ok
protection paid at default, % of notional            10.5055   book     10.5056   ok

upfront, legs differenced, %                          6.2797   book      6.2797   ok
upfront, short form (s-c)A, %                         6.2798   book     

## 2. 从利差到期初支付：例 25.2

**要看的是年金，而不是期初支付**。利差走阔时它从 4.55 降到 3.40，因为以正在恶化
的信用为标的的合约预计会提前终止，保费流预计走不完五年。正是这个变短
使得期初支付并不是利差的线性函数，第 3 节里交易台算错的也正是这一效应。


In [3]:
print("%12s %14s %16s %12s" % ("par spread", "hazard rate", "risky annuity", "upfront"))
for s_bp in (40, 100, 150, 250, 500, 800):
    s = s_bp / 1e4
    lam_s = s / (1 - R)
    A_s, _, _, _ = cds_legs(lam_s)
    print("%9d bp %13.3f%% %16.4f %11.2f%%"
          % (s_bp, 100 * lam_s, A_s, 100 * (s - coupon) * A_s))
print()
a_lo, _, _, _ = cds_legs(0.0040 / (1 - R))
a_hi, _, _, _ = cds_legs(0.0800 / (1 - R))
print("利差从 40 个基点走到 800 个基点时，年金从 %.2f 缩到 %.2f"
      % (a_lo, a_hi))


  par spread    hazard rate    risky annuity      upfront
       40 bp         0.667%           4.5513       -2.73%
      100 bp         1.667%           4.4428        0.00%
      150 bp         2.500%           4.3550        2.18%
      250 bp         4.167%           4.1865        6.28%
      500 bp         8.333%           3.8027       15.21%
      800 bp        13.333%           3.4044       23.83%

利差从 40 个基点走到 800 个基点时，年金从 4.55 缩到 3.40


## 3. 年金也在变：例 25.3

交易台当初在利差 120 个基点时按 100 个基点的标准票息买入五年保护；一年后四年期利差是 300 个基点。
盯市价值为 $(s_t - c)A_t$，其中 $c$ 是 100 个基点的票息，而不是成交时的 120 个基点；**右边两个对象都带下标 $t$**。用旧利差下的年金会高估
盈利，而这个误差的方向是系统性的：信用恶化时年金变短，因此保护的获利比
简单久期计算所显示的要*少*。


In [4]:
c = 0.0100                                   # the standard coupon the contract pays
A_new, _, _, _ = cds_legs(0.0300 / (1 - R), T=4.0)
A_old, _, _, _ = cds_legs(0.0120 / (1 - R), T=4.0)
mtm_new = (0.0300 - c) * A_new
mtm_old = (0.0300 - c) * A_old
mtm_traded = (0.0300 - 0.0120) * A_new

show("four-year annuity at the new spread", A_new, 3.4103)
show("MTM on the current curve, % of notional", 100 * mtm_new, 6.82)
show("MTM using the old-spread annuity, %", 100 * mtm_old, 7.22)
show("overstatement, %", 100 * (mtm_old / mtm_new - 1), 5.9, tol=3e-3)
show("marking against the traded 120 bp, %", 100 * mtm_traded, 6.14)
print()
print("按 1,000 万美元名义本金计，正确的盯市价值是 $%s" % format(NOTIONAL * mtm_new, ",.0f"))


four-year annuity at the new spread                   3.4103   book      3.4103   ok
MTM on the current curve, % of notional               6.8207   book      6.8200   ok
MTM using the old-spread annuity, %                   7.2236   book      7.2200   ok
overstatement, %                                      5.9076   book      5.9000   ok
marking against the traded 120 bp, %                  6.1386   book      6.1400   ok

按 1,000 万美元名义本金计，正确的盯市价值是 $682,067


## 4. 指数为什么不是平均：例 25.4

125 个主体，平均 80 个基点：115 个在 55 个基点，10 个困境主体在 367.5 个基点。
指数并不在 80 交易。**一件组合工具按其成分的年金加权平均定价**，而利差宽的主体
年金更短，因此它的权重比其利差所显示的要小。与实际成交水平的差额就是指数
偏斜；这里的 2.7 个基点是算术，不是定价错误。


In [5]:
BUCKETS = [(115, 0.0055), (10, 0.03675)]

weighted = total_annuity = 0.0
for count, s in BUCKETS:
    A_j, _, _, _ = cds_legs(s / (1 - R))
    weighted += count * A_j * s
    total_annuity += count * A_j

arithmetic = sum(count * s for count, s in BUCKETS) / sum(c for c, _ in BUCKETS)
show("arithmetic average, bp", 1e4 * arithmetic, 80.0)
show("annuity-weighted index spread, bp", 1e4 * weighted / total_annuity, 77.3)
print()
for count, s in BUCKETS:
    A_j, _, _, _ = cds_legs(s / (1 - R))
    print("%4d 个主体在 %6.1f 个基点，风险年金为 %.4f"
          % (count, 1e4 * s, A_j))


arithmetic average, bp                               80.0000   book     80.0000   ok
annuity-weighted index spread, bp                    77.3104   book     77.3000   ok

 115 个主体在   55.0 个基点，风险年金为 4.5239
  10 个主体在  367.5 个基点，风险年金为 3.9997


## 5. 同一个资产池，五个分层：例 25.5

125 个主体，每个五年违约概率 7%，回收率 40%。**无论相关性为多少，资产池的
预期损失都是 4.200%**：相关性只在各层之间重新分配损失，并不创造损失。

顺着列往下读，两端往相反方向走。危险在优先档：按 $\rho = 0.10$ 估值、实际面对
$\rho = 0.50$ 时，它的预期损失是**43 倍**，而资产池本身什么都没变。


In [6]:
NAMES, PD5 = 125, 0.07
LOSS_PER_DEFAULT = (1 - R) / NAMES
TRANCHES = [(0.00, 0.03), (0.03, 0.07), (0.07, 0.10), (0.10, 0.15), (0.15, 0.30)]

# 共同因子上的一张细网格，以标准正态密度为权。Gauss-Hermite 是教科书的选择，
# 在这里却是错的：节点超过两百个，其权重就溢出；低于此数时，
# 优先档（尾部的尾部）又尚未收敛。这张网格在一千点就已收敛。
_x = np.linspace(-8.0, 8.0, 4001)
_w = norm.pdf(_x)
_w = _w / _w.sum()
_k = np.arange(NAMES + 1)
_losses = _k * LOSS_PER_DEFAULT


def default_pmf(rho, p=PD5):
    """违约个数的无条件分布，单因子高斯 copula。"""
    if rho <= 0.0:
        return binom.pmf(_k, NAMES, p)
    c = norm.ppf(p)
    p_m = N((c - sqrt(rho) * _x) / sqrt(1 - rho))
    return (binom.pmf(_k[:, None], NAMES, p_m[None, :]) * _w[None, :]).sum(axis=1)


def tranche_el(pmf, a, b):
    """以分层自身名义本金为分母的预期损失。"""
    return (pmf * np.clip(_losses - a, 0.0, b - a)).sum() / (b - a)


print("%6s %10s %10s %10s %10s %10s %12s"
      % ("rho", "0-3%", "3-7%", "7-10%", "10-15%", "15-30%", "portfolio"))
for rho in (0.00, 0.10, 0.20, 0.30, 0.50, 0.80):
    pmf = default_pmf(rho)
    row = [100 * tranche_el(pmf, a, b) for a, b in TRANCHES]
    portfolio = 100 * (pmf * _losses).sum()
    print("%6.2f %9.2f%% %9.2f%% %9.2f%% %9.2f%% %9.2f%% %11.3f%%"
          % (rho, row[0], row[1], row[2], row[3], row[4], portfolio))
print()

print("%6s %24s %20s" % ("rho", "15-30% expected loss", "P(any loss at all)"))
for rho in (0.10, 0.30, 0.50, 0.80):
    pmf = default_pmf(rho)
    el = 100 * tranche_el(pmf, 0.15, 0.30)
    p_any = 100 * pmf[_losses > 0.15].sum()
    print("%6.2f %23.3f%% %19.2f%%" % (rho, el, p_any))
print()
el_10 = tranche_el(default_pmf(0.10), 0.15, 0.30)
el_50 = tranche_el(default_pmf(0.50), 0.15, 0.30)
print("按 rho = 0.10 估值、实际面对 rho = 0.50 时，优先档的预期损失")
print("是原来的 %.0f 倍，而资产池本身什么都没变"
      % (el_50 / el_10))


   rho       0-3%       3-7%      7-10%     10-15%     15-30%    portfolio
  0.00     95.76%     32.73%      0.59%      0.00%      0.00%       4.200%
  0.10     81.81%     33.49%      9.44%      2.16%      0.10%       4.200%
  0.20     71.40%     31.51%     13.50%      5.48%      0.78%       4.200%
  0.30     62.59%     29.30%     15.21%      7.87%      1.88%       4.200%
  0.50     47.36%     24.75%     15.81%     10.40%      4.27%       4.200%


  0.80     26.59%     17.11%     13.35%     10.75%      6.96%       4.200%

   rho     15-30% expected loss   P(any loss at all)


  0.10                   0.098%                0.61%
  0.30                   1.877%                4.97%
  0.50                   4.272%                7.93%
  0.80                   6.962%                9.44%



按 rho = 0.10 估值、实际面对 rho = 0.50 时，优先档的预期损失
是原来的 43 倍，而资产池本身什么都没变


## 6. 五个主体，五种工具：例 25.6

五个主体都在 150 个基点。**相关性为零时，首次违约篮子的价格等于各部分之和**：
五个 150 加起来就是 750，因为主体相互独立时，首次违约以各主体违约强度之和的强度到来，篮子相当于以一个风险高五倍的主体为参照的 CDS。相关性
升高时，第 5 节的跷跷板在另一种工具上重现：首次违约篮子下降，第五次违约篮子上升。

而且五者之和守恒。由于违约个数等于“至少发生 $k$ 次”这个示性函数对 $k$ 的求和，
五个保护端在任何相关性下都必须等于五个单一主体保护端之和，最后一行就是这个校验。


In [7]:
BASKET, S_NAME = 5, 0.0150
LAM_NAME = S_NAME / (1 - R)
_j = np.arange(BASKET + 1)


def at_least_k_by(t, rho, k):
    """P(到 t 时五个中至少 k 个已违约)，单因子高斯 copula。"""
    F = 1.0 - exp(-LAM_NAME * t)
    if rho <= 0.0:
        return binom.sf(k - 1, BASKET, F)
    c = norm.ppf(F)
    p_m = N((c - sqrt(rho) * _x) / sqrt(1 - rho))
    return float((_w * binom.sf(k - 1, BASKET, p_m)).sum())


def basket_spread(rho, k, T=TENOR, r=RATE, freq=FREQ):
    """第 k 违约篮子的平价利差，惯例与单一主体合约相同。"""
    n, dt = int(round(T * freq)), 1.0 / freq
    ann = prot = 0.0
    for i in range(1, n + 1):
        t, t_prev = i * dt, (i - 1) * dt
        G, G_prev = at_least_k_by(t, rho, k), at_least_k_by(t_prev, rho, k)
        df = exp(-r * t)
        dG = G - G_prev
        ann += dt * df * (1.0 - G) + 0.5 * dt * df * dG
        prot += df * dG
    return (1 - R) * prot / ann, (1 - R) * prot


print("%6s %8s %8s %8s %8s %8s" % ("rho", "1st", "2nd", "3rd", "4th", "5th"))
for rho in (0.00, 0.15, 0.30, 0.60, 0.90):
    spreads = [basket_spread(rho, k)[0] for k in range(1, BASKET + 1)]
    print("%6.2f %8.0f %8.0f %8.0f %8.0f %8.0f"
          % (rho, *[1e4 * s for s in spreads]))
print()

_, single_prot, _, _ = cds_legs(LAM_NAME)
print("%6s %22s %22s" % ("rho", "sum of basket legs", "5 single-name legs"))
for rho in (0.00, 0.30, 0.90):
    total = sum(basket_spread(rho, k)[1] for k in range(1, BASKET + 1))
    print("%6.2f %21.6f%% %21.6f%%" % (rho, 100 * total, 100 * BASKET * single_prot))


   rho      1st      2nd      3rd      4th      5th


  0.00      750      133       16        1        0
  0.15      669      160       34        5        0
  0.30      591      179       56       15        2
  0.60      439      196       99       47       17


  0.90      269      185      138      102       67

   rho     sum of basket legs     5 single-name legs
  0.00             32.662700%             32.662700%
  0.30             32.662700%             32.662700%
  0.90             32.662700%             32.662700%


## 7. 重组条款值多少钱：例 25.7

破产或支付违约的年强度为 3%，回收率 40%；把重组计入使强度再增加 15%，但重组事件
的回收率为 70%。

**要点不是这个数字，它完全取决于所假定的强度**。要点是：这个数字一直小而稳定，
直到它不再如此：一旦重组真正进入谈判，两种合约就不再是近似替代品了。


In [8]:
LAM_HARD, R_HARD = 0.03, 0.40
LAM_RESTR, R_RESTR = 0.03 * 0.15, 0.70

s_no = LAM_HARD * (1 - R_HARD)
s_with = s_no + LAM_RESTR * (1 - R_RESTR)

show("no-restructuring contract, bp", 1e4 * s_no, 180.0)
show("with restructuring, bp", 1e4 * s_with, 193.5)
show("value of the clause, bp", 1e4 * (s_with - s_no), 13.5)
show("as a share of the spread, %", 100 * (s_with - s_no) / s_with, 7.0, tol=5e-3)


no-restructuring contract, bp                       180.0000   book    180.0000   ok
with restructuring, bp                              193.5000   book    193.5000   ok
value of the clause, bp                              13.5000   book     13.5000   ok
as a share of the spread, %                           6.9767   book      7.0000   ok


## 8. 分解一只可转债：例 25.8

面值 1000，票息 2%，五年期，转股比例 20；股价 40，波动率 35%，利率 4%，
发行人的信用利差使债底按 7%的收益率贴现。

**可转债就是债底加上一份以 20 股为标的、执行价 50 的看涨期权**。随股价上行，它的 delta
从 6 股走到 17 股（股价低时像债，股价高时像股），而发行人卖掉的正是这个过渡。


In [9]:
FACE, CPN, T_CB, RATIO = 1000.0, 0.02, 5.0, 20
S0, VOL, R_F, Y_CREDIT = 40.0, 0.35, 0.04, 0.07
STRIKE = FACE / RATIO


def bond_floor(y=Y_CREDIT):
    """普通债价值，按发行人收益率连续贴现。"""
    return sum(FACE * CPN * exp(-y * t) for t in range(1, 6)) + FACE * exp(-y * T_CB)


def bs_call(S, K=STRIKE, r=R_F, sigma=VOL, T=T_CB):
    """Black-Scholes 看涨期权，返回价格与 N(d1)。"""
    d1 = (log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * N(d1) - K * exp(-r * T) * N(d2), N(d1)


floor = bond_floor()
call, nd1 = bs_call(S0)

show("bond floor", floor, 786.14)
show("conversion value (20 x 40)", RATIO * S0, 800.00)
show("conversion price (1000 / 20)", STRIKE, 50.00)
show("conversion premium over the share price, %", 100 * (STRIKE / S0 - 1), 25.0)
show("embedded call on 20 shares", RATIO * call, 237.13)
show("convertible value", floor + RATIO * call, 1023.28)
show("equity delta, shares", RATIO * nd1, 12.82)
print()

print("%12s %14s %12s %16s" % ("share price", "convertible", "parity", "delta (shares)"))
for S in (20, 30, 40, 50, 70):
    c, n1 = bs_call(float(S))
    print("%12d %14.2f %12.2f %16.2f"
          % (S, floor + RATIO * c, RATIO * S, RATIO * n1))


bond floor                                          786.1443   book    786.1400   ok
conversion value (20 x 40)                          800.0000   book    800.0000   ok
conversion price (1000 / 20)                         50.0000   book     50.0000   ok
conversion premium over the share price, %           25.0000   book     25.0000   ok
embedded call on 20 shares                          237.1347   book    237.1300   ok
convertible value                                  1023.2790   book   1023.2800   ok
equity delta, shares                                 12.8245   book     12.8200   ok

 share price    convertible       parity   delta (shares)
          20         827.87       400.00             6.00
          30         908.55       600.00             9.95
          40        1023.28       800.00            12.82
          50        1162.13      1000.00            14.82
          70        1484.96      1400.00            17.18


## 9. 在带违约的树上给可转债定价：例 25.9

例 25.8 的相加拆解只是初步估计，并非估值。第 25.14 节把债券、转股权、发行人的赎回权与
发行人的违约放在同一棵树上：每一步股价上行、下行，或发行人以概率
$1 - e^{-\lambda\Delta t}$ 违约，此时股票一文不值，债券值其回收额。每个节点的倒推为
$V = \max\{nS,\ \min(\text{继续持有}, K_c)\}$ 再加当期票息。

**三点解读**。违约分枝让内嵌的看涨期权变*贵*而不是变便宜（Merton 的“跳跃至破产”结果：
按 $r+\lambda$ 计算的布莱克–斯科尔斯–默顿价格）；没有股息时美式转股权一文不值；
发行人的赎回权是最大的一项。本单元复现图 25.4 的五步树、500 步的收敛值，以及例中完整
算出的三个节点。


In [10]:
LAM_CB = 0.03 / (1 - 0.40)          # hazard rate from the credit triangle: s / (1 - R)
R_CB, K_CALL, CALL_FROM = 0.40, 1100.0, 2.0


def cb_tree(n, conv="american", call_price=None, call_from=CALL_FROM, trace=None):
    """Convertible on a tree where each step is up, down, or default (Hull's construction).

    conv: "none" (bond floor), "european" (convert only at maturity), "american".
    Rollback: V = max{ ratio*S, min(continuation, call_price) } + coupon due.
    trace: optional (step, S) -- returns that node's continuation value instead.
    """
    dt = T_CB / n
    u = exp(sqrt((VOL ** 2 - LAM_CB) * dt))
    d = 1.0 / u
    a = exp(R_F * dt)
    p_def = 1.0 - exp(-LAM_CB * dt)
    p_u = (a - d * exp(-LAM_CB * dt)) / (u - d)
    p_d = (u * exp(-LAM_CB * dt) - a) / (u - d)
    disc = exp(-R_F * dt)
    V = []
    for j in range(n + 1):
        S = S0 * u ** (n - 2 * j)
        V.append((max(RATIO * S, FACE) if conv != "none" else FACE) + CPN * FACE)
    for i in range(n - 1, -1, -1):
        t = i * dt
        coupon = CPN * FACE if (abs(t - round(t)) < 1e-9 and i > 0) else 0.0
        new_V = []
        for j in range(i + 1):
            S = S0 * u ** (i - 2 * j)
            cont = disc * (p_u * V[j] + p_d * V[j + 1] + p_def * R_CB * FACE)
            if trace is not None and i == trace[0] and abs(S - trace[1]) < 0.01:
                return cont
            conv_value = RATIO * S if conv != "none" else 0.0
            held = cont
            if call_price is not None and t >= call_from - 1e-9:
                held = max(min(cont, call_price), conv_value)
            v = max(held, conv_value) if conv == "american" else held
            new_V.append(v + coupon)
        V = new_V
    return V[0], dict(u=u, d=d, p_u=p_u, p_d=p_d, p_def=p_def, disc=disc)


_, prm = cb_tree(5)
for k, printed in (("u", 1.3090), ("d", 0.7639), ("p_u", 0.5763), ("p_d", 0.3749), ("p_def", 0.0488), ("disc", 0.9608)):
    show("five-step tree: " + k, prm[k], printed, tol=2e-3)
print()

print("%-52s %10s %10s" % ("", "5 steps", "500 steps"))
for label, kw, printed in (("bond floor (no conversion right)", dict(conv="none"), (793.50, 795.10)),
                           ("conversion at maturity only", dict(conv="european"), (1046.38, 1057.82)),
                           ("conversion at any time", dict(conv="american"), (1046.38, 1057.82)),
                           ("conversion at any time, callable at 1100 from year 2",
                            dict(conv="american", call_price=K_CALL), (994.47, 988.40))):
    v5, v500 = cb_tree(5, **kw)[0], cb_tree(500, **kw)[0]
    flag = "ok" if abs(v5 - printed[0]) < 0.01 and abs(v500 - printed[1]) < 0.01 else "DIFFERS"
    print("%-52s %10.2f %10.2f   %s" % (label, v5, v500, flag))
print()

show("node t=4, S=40: continuation", cb_tree(5, "american", K_CALL, trace=(4, 40.0)), 977.09)
show("node t=2, S=68.54: continuation (issuer calls, holder converts)", cb_tree(5, "american", K_CALL, trace=(2, 68.54)), 1426.81)
show("node t=3, S=52.36: continuation (issuer calls, holder takes 1100)", cb_tree(5, "american", K_CALL, trace=(3, 52.36)), 1148.00)


five-step tree: u                                     1.3090   book      1.3090   ok
five-step tree: d                                     0.7639   book      0.7639   ok
five-step tree: p_u                                   0.5763   book      0.5763   ok
five-step tree: p_d                                   0.3749   book      0.3749   ok
five-step tree: p_def                                 0.0488   book      0.0488   ok
five-step tree: disc                                  0.9608   book      0.9608   ok

                                                        5 steps  500 steps
bond floor (no conversion right)                         793.50     795.10   ok
conversion at maturity only                             1046.38    1057.82   ok


conversion at any time                                  1046.38    1057.82   ok
conversion at any time, callable at 1100 from year 2     994.47     988.40   ok

node t=4, S=40: continuation                        977.0859   book    977.0900   ok
node t=2, S=68.54: continuation (issuer calls, holder converts)     1426.8132   book   1426.8100   ok
node t=3, S=52.36: continuation (issuer calls, holder takes 1100)     1148.0039   book   1148.0000   ok


## 10. 混合结构需要、而市场没有报价的相关性：例 25.10

一份票据，在到期时支付一笔票息，条件是主体存续**且**某股票指数收于水平 $K$ 之上。两端各自可观察，
而它们的相依性不可观察，全部的困难就在于此。

指数端以票据的初始水平为执行价，因此它的边缘概率由所给参数决定：在 4\% 的利率、2\% 的股息率
与 30\% 的波动率、五年期限下，$N(d_2) = 0.4261$，并且在这里是算出来的，不是选定的。有意思的
量是**联合**概率。它是一个二元正态概率。本单元在采信其中间取值之前，先检验任何此类计算都必须
满足的三个条件：$\rho = 0$ 必须还原两个边缘概率的乘积，而 $\rho \to \pm 1$ 必须达到 Fréchet 界。

In [11]:
from scipy.stats import multivariate_normal

P_SURV, COUPON = 0.90, 8.0

# --- the index leg is struck at its initial level; its marginal follows from r, q and sigma
R_X, Q_X, SIG_X, T_X = 0.04, 0.02, 0.30, 5.0
D2_X = (R_X - Q_X - 0.5 * SIG_X ** 2) * T_X / (SIG_X * np.sqrt(T_X))
P_UP = N(D2_X)
show("d_2 for the index leg", D2_X, -0.1863)
show("P(index above its initial level)", P_UP, 0.4261)

# --- the joint probability, and the four controls that make it trustworthy
def both(rho, p_surv=P_SURV, p_up=P_UP):
    """P(survive AND index above) under a Gaussian copula on the two latent factors."""
    c, k = norm.ppf(1 - p_surv), norm.ppf(1 - p_up)
    cov = [[1.0, rho], [rho, 1.0]]
    # P(X > c, Y > k) by the survival function of the bivariate normal
    return multivariate_normal(mean=[0.0, 0.0], cov=cov).cdf([-c, -k])


INDEP = P_SURV * P_UP
show("control: rho = 0 gives the product of marginals", both(0.0), INDEP)
show("control: rho -> +1 gives min(p1, p2)", both(0.999), min(P_SURV, P_UP), tol=0.01)
show("control: rho -> -1 gives max(p1+p2-1, 0)", both(-0.999), max(P_SURV + P_UP - 1.0, 0.0), tol=0.01)
print()
print("%-8s %16s %18s %22s" % ("rho", "P(coupon paid)", "expected coupon", "effect on the coupon"))
for rho, p_book, cpn_book in ((-0.4, 0.3551, 2.8408), (0.0, 0.3835, 3.0678), (0.4, 0.4091, 3.2727)):
    p = both(rho)
    show("rho %+0.1f, P(coupon paid)" % rho, p, p_book)
    show("rho %+0.1f, expected coupon %%" % rho, COUPON * p, cpn_book)
    bp = 1e4 * COUPON * (p - INDEP) / 100.0
    print("%-8.1f %16.4f %18.4f %19.1f bp" % (rho, p, COUPON * p, bp))

# --- the band any dependence must respect, which is what makes a reserve defensible
lo, hi = max(P_SURV + P_UP - 1.0, 0.0), min(P_SURV, P_UP)
show("the Frechet band, bp of coupon", 1e4 * COUPON * (hi - lo) / 100.0, 80.0)
print()
print("无论相依性如何，票息都落在 [%.4f%%, %.4f%%] 之内，即一个宽 %.0f bp 的区间。"
      % (COUPON * lo, COUPON * hi, 1e4 * COUPON * (hi - lo) / 100.0))
print("按这个宽度计提准备，无须就相关性本身达成一致。")

d_2 for the index leg                                -0.1863   book     -0.1863   ok
P(index above its initial level)                      0.4261   book      0.4261   ok
control: rho = 0 gives the product of marginals        0.3835   book      0.3835   ok
control: rho -> +1 gives min(p1, p2)                  0.4261   book      0.4261   ok
control: rho -> -1 gives max(p1+p2-1, 0)              0.3261   book      0.3261   ok

rho        P(coupon paid)    expected coupon   effect on the coupon
rho -0.4, P(coupon paid)                              0.3551   book      0.3551   ok
rho -0.4, expected coupon %                           2.8408   book      2.8408   ok
-0.4               0.3551             2.8408               -22.7 bp
rho +0.0, P(coupon paid)                              0.3835   book      0.3835   ok
rho +0.0, expected coupon %                           3.0678   book      3.0678   ok
0.0                0.3835             3.0678                -0.0 bp
rho +0.4, P(coupon paid)     